# 🎬 GECA — Generar Video Anotado
Genera video MP4 con bounding boxes. Auto-comprime a H.264 para streaming en la app.

In [ ]:
!pip install ultralytics -q

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

## 1. Configuración

In [ ]:
import os

SHARED = '/home/jovyan/shared'
MODEL_PATH = f'{SHARED}/models/geca_Bu_Bublik_v2_v1_best.pt'
VIDEO_PATH = f'{SHARED}/videos/Bu_Bublik_v2.mp4'

OUTPUT_RESOLUTION = 480     # 480 = streaming rápido, 720 = HD
CONF_THRESHOLD = 0.25
IMG_SIZE = 640
BATCH_SIZE = 32
OUTPUT_DIR = f'{SHARED}/results'
LINE_WIDTH = 2
FONT_SCALE = 0.6
CRF = 28                    # Compresión H.264 (23=alta calidad, 28=buena, 35=baja)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Video: {os.path.basename(VIDEO_PATH)}')
print(f'Salida: {OUTPUT_RESOLUTION}p, CRF: {CRF}')

## 2. Cargar Modelo y Analizar

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO(MODEL_PATH)
class_names = model.names
print(f'Clases: {list(class_names.values())}')

cap = cv2.VideoCapture(VIDEO_PATH)
orig_fps = cap.get(cv2.CAP_PROP_FPS)
orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / orig_fps
cap.release()

OUTPUT_FPS = orig_fps
ratio = min(1, OUTPUT_RESOLUTION / orig_h)
out_w = int(orig_w * ratio) if int(orig_w * ratio) % 2 == 0 else int(orig_w * ratio) + 1
out_h = int(orig_h * ratio) if int(orig_h * ratio) % 2 == 0 else int(orig_h * ratio) + 1
frame_interval = max(1, int(orig_fps / OUTPUT_FPS))
frames_to_process = total_frames // frame_interval

print(f'Original: {orig_w}x{orig_h} @ {orig_fps:.0f}fps, {duration/60:.1f}min')
print(f'Salida: {out_w}x{out_h} @ {OUTPUT_FPS:.0f}fps, {frames_to_process} frames')

## 3. Generar Video

In [ ]:
import time, subprocess
from collections import defaultdict
from tqdm.notebook import tqdm

video_name = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
temp_path = os.path.join(OUTPUT_DIR, f'{video_name}_temp.mp4')
output_path = os.path.join(OUTPUT_DIR, f'{video_name}_anotado_{OUTPUT_RESOLUTION}p.mp4')

COLORS = [(108,92,231),(0,184,148),(225,112,85),(255,212,59),(59,130,246),(239,68,68),(16,185,129),(245,158,11)]

cap = cv2.VideoCapture(VIDEO_PATH)
writer = cv2.VideoWriter(temp_path, cv2.VideoWriter_fourcc(*'mp4v'), OUTPUT_FPS, (out_w, out_h))

class_det = defaultdict(int)
class_fc = defaultdict(int)
processed = 0
idx = 0
batch_frames = []
t0 = time.time()
pbar = tqdm(total=frames_to_process, desc='Procesando', unit='f')

def draw_and_write(batch):
    global processed
    if not batch: return
    results = model(batch, conf=CONF_THRESHOLD, imgsz=IMG_SIZE, verbose=False)
    sx, sy = out_w/orig_w, out_h/orig_h
    for result, frame in zip(results, batch):
        out = cv2.resize(frame, (out_w, out_h), interpolation=cv2.INTER_AREA) if ratio < 1 else frame.copy()
        fc = set()
        for box in result.boxes:
            cid = int(box.cls.item())
            conf = float(box.conf.item())
            x1,y1,x2,y2 = [int(v) for v in (box.xyxy[0].cpu().numpy() * [sx,sy,sx,sy])]
            color = COLORS[cid % len(COLORS)]
            cn = class_names.get(cid, str(cid))
            cv2.rectangle(out, (x1,y1), (x2,y2), color, LINE_WIDTH)
            label = f'{cn} {conf:.0%}'
            (tw,th),_ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, 1)
            cv2.rectangle(out, (x1,y1-th-8), (x1+tw+4,y1), color, -1)
            cv2.putText(out, label, (x1+2,y1-4), cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, (255,255,255), 1, cv2.LINE_AA)
            class_det[cn] += 1
            fc.add(cn)
        for cn in fc: class_fc[cn] += 1
        writer.write(out)
    processed += len(batch)
    pbar.update(len(batch))

while True:
    ret, frame = cap.read()
    if not ret: break
    if idx % frame_interval == 0:
        batch_frames.append(frame)
        if len(batch_frames) >= BATCH_SIZE:
            draw_and_write(batch_frames)
            batch_frames = []
    idx += 1
draw_and_write(batch_frames)
pbar.close()
cap.release()
writer.release()

print('\nComprimiendo a H.264...')
subprocess.run(['ffmpeg', '-i', temp_path, '-c:v', 'libx264', '-crf', str(CRF),
    '-preset', 'fast', '-movflags', '+faststart', '-y', output_path],
    capture_output=True)
os.remove(temp_path)

elapsed = time.time() - t0
mb = os.path.getsize(output_path) / 1024 / 1024
print(f'\n✓ {output_path}')
print(f'  {mb:.0f} MB, {elapsed/60:.1f} min, {processed/elapsed:.0f} f/s')
print(f'\n📂 Windows: \\\\10.43.13.186\\Compartida\\results\\')

## 4. Resumen de Detecciones

In [ ]:
print(f'{"Etiqueta":<20} {"Det":>10} {"Frames":>10} {"Tiempo(s)":>10} {"%":>8}')
print(f'{"-"*62}')
for cn in sorted(class_det, key=lambda x: -class_det[x]):
    t = class_fc[cn]/OUTPUT_FPS
    p = class_fc[cn]/processed*100 if processed else 0
    print(f'{cn:<20} {class_det[cn]:>10,} {class_fc[cn]:>10,} {t:>10.1f} {p:>8.2f}')